# 05 · Professional Workflow: Project Layout, Logging Config, Settings, pre-commit, CI/CD, Docker & Docs

**Module 07 – Errors, Debugging & Testing** · Estimated time: 60 min

> ❗ Neither source covers how real Python projects are set up and shipped. This lesson ties together tools from Modules 07, 08 and 16.

### 🎯 What you will learn
- A production-ready project layout
- **Logging configuration** with `dictConfig`, handlers, rotation, and **structured (JSON) logs**
- Configuration & secrets: environment variables, `.env` files, 12-factor apps (and Pydantic settings, Module 15)
- Code quality automation: **Ruff**, **mypy**, **pre-commit** hooks
- **CI** with GitHub Actions (lint, type-check, test matrix, coverage)
- Packaging an app in **Docker**
- Documentation: docstrings, MkDocs / Sphinx
- Security hygiene: dependency audits, `bandit`, secret scanning

## 1. A production-ready layout

```
weather-service/
├── .github/workflows/ci.yml       ← CI pipeline
├── .pre-commit-config.yaml        ← git hooks
├── .gitignore  .python-version  .env.example
├── Dockerfile  .dockerignore
├── pyproject.toml                 ← metadata, deps, [tool.ruff], [tool.mypy], [tool.pytest...]
├── uv.lock                        ← locked dependencies (Module 08 · 04)
├── README.md  CHANGELOG.md  LICENSE
├── docs/                          ← MkDocs/Sphinx sources
├── src/weather_service/
│   ├── __init__.py
│   ├── __main__.py                ← `python -m weather_service`
│   ├── config.py                  ← settings from the environment
│   ├── logging_config.py
│   ├── domain/                    ← business logic (no I/O)
│   ├── services/                  ← use cases
│   ├── adapters/                  ← DB, HTTP clients, files (Module 10 · 08)
│   └── api/                       ← CLI / FastAPI routes
└── tests/
    ├── conftest.py  unit/  integration/
```

## 2. Logging configuration with `dictConfig`

Module 07 · 02 introduced `logging`. Real apps configure it **once**, at start-up, from a dict (or file):

In [1]:
import logging, logging.config, sys, json

LOGGING = {
    "version": 1,
    "disable_existing_loggers": False,
    "formatters": {
        "console": {"format": "%(asctime)s %(levelname)-8s %(name)s: %(message)s", "datefmt": "%H:%M:%S"},
    },
    "handlers": {
        "console": {"class": "logging.StreamHandler", "formatter": "console", "stream": "ext://sys.stdout",
                    "level": "INFO"},
        "file": {"class": "logging.handlers.RotatingFileHandler", "filename": "app.log",
                 "maxBytes": 1_000_000, "backupCount": 3, "formatter": "console", "level": "DEBUG",
                 "encoding": "utf-8"},
    },
    "loggers": {
        "weather": {"level": "DEBUG", "handlers": ["console", "file"], "propagate": False},
        "urllib3": {"level": "WARNING"},          # silence noisy libraries
    },
    "root": {"level": "WARNING", "handlers": ["console"]},
}
logging.config.dictConfig(LOGGING)

log = logging.getLogger("weather.api")             # child of "weather" → inherits its handlers
log.debug("only in the file")
log.info("service started on port %d", 8000)
log.warning("cache miss for %s", "Pune")
print("--- app.log:")
print(open("app.log", encoding="utf-8").read())

19:42:36 INFO     weather.api: service started on port 8000


19:42:36 WARNING  weather.api: cache miss for Pune


--- app.log:
19:42:36 DEBUG    weather.api: only in the file
19:42:36 INFO     weather.api: service started on port 8000
19:42:36 WARNING  weather.api: cache miss for Pune



Logger hierarchy: `weather.api` → `weather` → root. Handlers decide **where** (console, file, syslog, HTTP); formatters decide **how**; levels filter at both logger and handler.

### Structured (JSON) logs

Log aggregators (ELK, Loki, Datadog, CloudWatch) prefer one JSON object per line:

In [2]:
class JSONFormatter(logging.Formatter):
    def format(self, record):
        payload = {
            "time": self.formatTime(record, "%Y-%m-%dT%H:%M:%S"),
            "level": record.levelname,
            "logger": record.name,
            "message": record.getMessage(),
            **getattr(record, "context", {}),            # extra fields
        }
        if record.exc_info:
            payload["exception"] = self.formatException(record.exc_info)
        return json.dumps(payload)

handler = logging.StreamHandler(sys.stdout)
handler.setFormatter(JSONFormatter())
jlog = logging.getLogger("orders")
jlog.handlers, jlog.propagate = [handler], False
jlog.setLevel(logging.INFO)

jlog.info("order placed", extra={"context": {"order_id": 1042, "amount": 499.0, "user": "asha"}})
try:
    1 / 0
except ZeroDivisionError:
    jlog.exception("payment failed", extra={"context": {"order_id": 1042}})

{"time": "2026-10-20T19:42:36", "level": "INFO", "logger": "orders", "message": "order placed", "order_id": 1042, "amount": 499.0, "user": "asha"}


{"time": "2026-10-20T19:42:36", "level": "ERROR", "logger": "orders", "message": "payment failed", "order_id": 1042, "exception": "Traceback (most recent call last):\n  File \"/var/folders/96/f6xk04z91_gdystl3qfqv33r0000gn/T/ipykernel_5524/2179573231.py\", line 22, in <module>\n    1 / 0\n    ~~^~~\nZeroDivisionError: division by zero"}


> 💡 Libraries like **structlog** and **loguru** make structured logging even easier. Never log secrets (passwords, tokens, full card numbers).

## 3. Configuration & secrets (the 12-factor way)

- Keep config that changes between environments (dev/staging/prod) in **environment variables**, not code
- Commit a **`.env.example`** with dummy values; keep the real **`.env`** out of Git
- Validate config at start-up and fail fast

In [3]:
import os
from dataclasses import dataclass

@dataclass(frozen=True)
class Settings:
    env: str
    database_url: str
    debug: bool
    workers: int

    @classmethod
    def from_env(cls) -> "Settings":
        missing = [k for k in ("DATABASE_URL",) if k not in os.environ]
        if missing:
            raise RuntimeError(f"missing required environment variables: {missing}")
        return cls(
            env=os.getenv("APP_ENV", "dev"),
            database_url=os.environ["DATABASE_URL"],
            debug=os.getenv("DEBUG", "false").lower() in {"1", "true", "yes"},
            workers=int(os.getenv("WORKERS", "2")),
        )

try:
    Settings.from_env()
except RuntimeError as e:
    print("RuntimeError:", e)

os.environ.update(DATABASE_URL="postgresql://localhost/app", DEBUG="true")
print(Settings.from_env())

RuntimeError: missing required environment variables: ['DATABASE_URL']
Settings(env='dev', database_url='postgresql://localhost/app', debug=True, workers=2)


Loading a `.env` file: `uv run --env-file .env ...` (Module 08 · 04), `python-dotenv`, or **`pydantic-settings`**, which validates types automatically (Module 15 · 09).

## 4. Code quality automation

### Ruff + mypy configuration in `pyproject.toml`

```toml
[tool.ruff]
line-length = 100
target-version = "py312"
src = ["src", "tests"]

[tool.ruff.lint]
select = ["E", "W", "F", "I", "UP", "B", "SIM", "C4", "PT", "RUF", "S"]   # pycodestyle, pyflakes, isort, pyupgrade, bugbear, simplify, comprehensions, pytest-style, ruff, bandit-security
ignore = ["E501"]
[tool.ruff.lint.per-file-ignores]
"tests/*" = ["S101"]            # allow assert in tests

[tool.ruff.format]
quote-style = "double"

[tool.mypy]
strict = true
files = ["src"]
```

### pre-commit: run checks automatically before every commit

`.pre-commit-config.yaml`:
```yaml
repos:
  - repo: https://github.com/pre-commit/pre-commit-hooks
    rev: v6.0.0
    hooks:
      - id: trailing-whitespace
      - id: end-of-file-fixer
      - id: check-yaml
      - id: check-added-large-files
      - id: detect-private-key
  - repo: https://github.com/astral-sh/ruff-pre-commit
    rev: v0.17.1
    hooks:
      - id: ruff-check
        args: [--fix]
      - id: ruff-format
  - repo: https://github.com/pre-commit/mirrors-mypy
    rev: v2.4.0
    hooks:
      - id: mypy
```
```bash
uv tool install pre-commit      # or: pipx install pre-commit
pre-commit install              # installs the git hook
pre-commit run --all-files      # run on everything once
pre-commit autoupdate           # bump hook versions
```
> 💡 Hook `rev:` values above are examples; run `pre-commit autoupdate` to pin the latest. (**prek** is a faster drop-in re-implementation of pre-commit.)

### Seeing Ruff's security rules (`S`, from bandit) catch real issues

In [4]:
import subprocess, pathlib, textwrap
pathlib.Path("risky.py").write_text(textwrap.dedent('''
    import subprocess, pickle, hashlib, requests
    PASSWORD = "hunter2"
    def run(cmd):
        subprocess.call(cmd, shell=True)
    def load(blob):
        return pickle.loads(blob)
    def digest(data):
        return hashlib.md5(data).hexdigest()
    def fetch(url):
        return requests.get(url)
'''))
out = subprocess.run([sys.executable, "-m", "ruff", "check", "risky.py", "--select", "S",
                      "--no-cache", "--output-format", "concise"], capture_output=True, text=True).stdout
print(out)

risky.py:3:12: S105 Possible hardcoded password assigned to: "PASSWORD"
risky.py:5:5: S602 `subprocess` call with `shell=True` identified, security issue
risky.py:7:12: S301 `pickle` and modules that wrap it can be unsafe when used to deserialize untrusted data, possible security issue
risky.py:9:12: S324 Probable use of insecure hash functions in `hashlib`: `md5`
risky.py:11:12: S113 Probable use of `requests` call without timeout
Found 5 errors.



## 5. Continuous Integration: GitHub Actions

`.github/workflows/ci.yml`:
```yaml
name: CI
on:
  push: { branches: [main] }
  pull_request:

jobs:
  lint:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v5
      - uses: astral-sh/setup-uv@v7
      - run: uv sync --locked --all-groups
      - run: uv run ruff check . && uv run ruff format --check .
      - run: uv run mypy

  test:
    runs-on: ${{ matrix.os }}
    strategy:
      fail-fast: false
      matrix:
        os: [ubuntu-latest, windows-latest, macos-latest]
        python: ["3.12", "3.13", "3.14", "3.15"]
    steps:
      - uses: actions/checkout@v5
      - uses: astral-sh/setup-uv@v7
        with: { python-version: "${{ matrix.python }}" }
      - run: uv sync --locked
      - run: uv run pytest --cov --cov-report=xml
      - uses: codecov/codecov-action@v5
        if: matrix.os == 'ubuntu-latest' && matrix.python == '3.15'

  audit:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v5
      - uses: astral-sh/setup-uv@v7
      - run: uvx pip-audit        # or: uv audit (preview)
```
Add **branch protection** so PRs can only merge when CI is green. Releases are handled by a separate workflow (Module 08 · 07, trusted publishing). **Dependabot/Renovate** open PRs to update dependencies.

## 6. Shipping a service in Docker

```dockerfile
# syntax=docker/dockerfile:1
FROM python:3.15-slim AS base
COPY --from=ghcr.io/astral-sh/uv:0.13 /uv /bin/uv
ENV PYTHONDONTWRITEBYTECODE=1 PYTHONUNBUFFERED=1 UV_COMPILE_BYTECODE=1 UV_LINK_MODE=copy
WORKDIR /app

COPY pyproject.toml uv.lock ./
RUN uv sync --locked --no-dev --no-install-project      # cached dependency layer

COPY src ./src
RUN uv sync --locked --no-dev

RUN useradd --create-home appuser                         # don't run as root
USER appuser
EXPOSE 8000
HEALTHCHECK CMD python -c "import urllib.request; urllib.request.urlopen('http://localhost:8000/health')"
CMD ["/app/.venv/bin/uvicorn", "weather_service.api:app", "--host", "0.0.0.0", "--port", "8000"]
```
`.dockerignore`: `.venv`, `.git`, `__pycache__`, `.env`, `tests`, `dist`.

```bash
docker build -t weather-service .
docker run --env-file .env -p 8000:8000 weather-service
```

## 7. Documentation

- **Docstrings** (Google/NumPy style) on public functions/classes; type hints document types
- **README**: what it is, install, quick-start, configuration, contributing
- **MkDocs + Material** (Markdown) with **mkdocstrings** (API docs from docstrings), or **Sphinx** (reStructuredText, used by CPython)
- Host on GitHub Pages / Read the Docs
- Keep a **CHANGELOG** and architecture decision records (ADRs) for important choices

```bash
uv add --group docs mkdocs-material "mkdocstrings[python]"
uv run mkdocs new . && uv run mkdocs serve
```

## 8. Security hygiene checklist
- ✅ Pin & lock dependencies; audit them (`uv audit`, `pip-audit`); enable Dependabot
- ✅ Lint for security issues (Ruff `S` rules / `bandit`)
- ✅ Never commit secrets: `.gitignore`, pre-commit `detect-private-key`, GitHub secret scanning; rotate leaked keys
- ✅ Validate all external input (Pydantic, Module 15); parameterised SQL (Module 11 · 03, Module 14 · 06)
- ✅ Don't `pickle`/`eval` untrusted data; use `secrets` for tokens (Module 12 · 03)
- ✅ Run containers as non-root; keep base images updated
- ✅ Use HTTPS with verification; set timeouts on network calls

## 📝 Summary
- Use a src layout with clear layers, configuration via environment variables, and logging set up once with `dictConfig` (JSON logs in production).
- Automate quality with Ruff, mypy, pytest and pre-commit; enforce it in CI on every PR.
- Ship services as small, non-root Docker images with locked dependencies; document with README + MkDocs.

## 🏋️ Practice
1. Add a `logging_config.py` to one of your projects with console + rotating file handlers.
2. Create `.pre-commit-config.yaml` with Ruff and run `pre-commit run --all-files`.
3. Write a GitHub Actions workflow that runs `ruff`, `mypy` and `pytest` on Python 3.13–3.15.
4. Write a Dockerfile for your Module 18 to-do CLI and run it with `docker run`.